# JAXTPC Simulation

GPU-accelerated liquid argon TPC detector simulation.

**Workflow:**
1. Load detector configuration and particle segment data
2. Run simulation (recombination, drift, diffusion, wire response)
3. Process response: noise + threshold + sparse decomposition
4. Visualize: detector response, truth hits, track labels

**Settings:** Toggle `INCLUDE_NOISE`, `INCLUDE_TRACK_HITS`, and `THRESHOLD_ENC` in the configuration cell.

## Setup and Configuration

In [ ]:
# =============================================================================
# CONFIGURATION - Modify these paths and parameters as needed
# =============================================================================

# Path to detector configuration YAML file
CONFIG_PATH = "config/cubic_wireplane_config.yaml"

# Path to HDF5 file containing particle segment data
# DATA_PATH = "muon.h5"
# Load for higher activity event
DATA_PATH = "out.h5"

# Event index to simulate (0-indexed)
EVENT_IDX = 7

# Number of diffusion kernel interpolation levels
# Higher values = smoother diffusion but slower compilation
NUM_S = 16

# Path to pre-computed wire response kernels
RESPONSE_PATH = "tools/responses/"

# Use bucketed (sparse) accumulation for memory efficiency
# Set to False for dense accumulation (uses more memory but simpler)
USE_BUCKETED = True

# Maximum active buckets per plane (only used if USE_BUCKETED=True)
MAX_ACTIVE_BUCKETS = 1000

# --- Single threshold for both track labeling and noise ---
THRESHOLD_ENC = 1200           # Deadband threshold in electrons
INCLUDE_NOISE = True           # Add intrinsic noise to detector response

INCLUDE_ELECTRONICS_RESPONSE = True
l
INCLUDE_ELECTRIC_DISTORTIONS = True

INCLUDE_DIGITIZATION = True    # Apply ADC digitization (quantize + clamp)

# --- Track labeling (hit path) ---
INCLUDE_TRACK_HITS = True      # Run hit path for truth-level track labeling

print("Configuration:")
print(f"  Config: {CONFIG_PATH}")
print(f"  Data: {DATA_PATH}")
print(f"  Event: {EVENT_IDX}")
print(f"  Bucketed mode: {USE_BUCKETED}")
print(f"  Threshold: {THRESHOLD_ENC} e-")
print(f"  Noise: {'ON' if INCLUDE_NOISE else 'OFF'}")
print(f"  Digitization: {'ON' if INCLUDE_DIGITIZATION else 'OFF'}")
print(f"  Track labeling: {'ON' if INCLUDE_TRACK_HITS else 'OFF'}")

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import gc

# Core simulation components
from tools.simulation import DetectorSimulator
from tools.config import create_diffusion_params, create_track_hits_config
from tools.geometry import generate_detector
from tools.loader import load_particle_step_data

# Noise pipeline
from tools.noise import process_response, extract_signal

# Visualization
from tools.visualization import (
    visualize_wire_signals,
    visualize_diffused_charge,
    visualize_track_labels,
    get_top_tracks_by_charge,
)

# Create output directory for plots
os.makedirs("plots", exist_ok=True)

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

## Load Data

Load detector configuration and particle segment data from HDF5 file.

**Note**: Input segments are energy deposits along particle tracks (e.g., from Geant4).
Each segment has a position, energy deposit, track angle, and track ID.

In [ ]:
# =============================================================================
# LOAD DETECTOR CONFIGURATION
# =============================================================================

# Load detector geometry and parameters from YAML config
detector_config = generate_detector(CONFIG_PATH)

# Create diffusion parameters from detector config
diffusion_params = create_diffusion_params(
    max_sigma_trans_unitless=detector_config['max_sigma_trans_unitless'],
    max_sigma_long_unitless=detector_config['max_sigma_long_unitless'],
    num_s=NUM_S,
    n_sigma=3.0
)

# Track labeling uses same threshold as noise pipeline
track_config = create_track_hits_config(threshold=THRESHOLD_ENC, max_tracks=30000) if INCLUDE_TRACK_HITS else None

print("Detector configuration loaded")
print(f"  Diffusion kernel half-widths: K_wire={diffusion_params.K_wire}, K_time={diffusion_params.K_time}")
if INCLUDE_TRACK_HITS:
    print(f"  Track labeling threshold: {THRESHOLD_ENC} e-")

In [ ]:
# =============================================================================
# LOAD PARTICLE SEGMENT DATA
# =============================================================================

print(f"Loading event {EVENT_IDX} from {DATA_PATH}...")

deposit_data = load_particle_step_data(DATA_PATH, EVENT_IDX)

n_segments = deposit_data.positions_mm.shape[0]
n_unique_tracks = len(jnp.unique(deposit_data.track_ids))
print(f"Loaded {n_segments:,} particle segments")
print(f"Unique tracks: {n_unique_tracks:,}")

# Show east/west split (dual-sided TPC)
x_mm = deposit_data.positions_mm[:, 0]
n_east = int(jnp.sum(x_mm < 0))
n_west = int(jnp.sum(x_mm >= 0))
print(f"\nSide distribution:")
print(f"  East (x<0):  {n_east:,} segments")
print(f"  West (x>=0): {n_west:,} segments")

## Create Simulator and Run

Initialize the detector simulator and process the event.

The simulator produces two outputs:
1. **track_hits**: Truth-level hits with track_id preserved
2. **response_signals**: Detector response after wire response convolution

In [ ]:
# =============================================================================
# CREATE DETECTOR SIMULATOR
# =============================================================================

jax.clear_caches()
gc.collect()

print("Creating DetectorSimulator...")

simulator = DetectorSimulator(
    detector_config,
    response_path=RESPONSE_PATH,
    track_config=track_config,
    diffusion_params=diffusion_params,
    use_bucketed=USE_BUCKETED,
    max_active_buckets=MAX_ACTIVE_BUCKETS if USE_BUCKETED else None,
    include_noise=INCLUDE_NOISE,
    include_electronics=INCLUDE_ELECTRONICS_RESPONSE,
    include_track_hits=INCLUDE_TRACK_HITS,
    include_electric_dist=INCLUDE_ELECTRIC_DISTORTIONS,
    include_digitize=INCLUDE_DIGITIZATION,
)

print("\nWarming up JIT (this may take a minute on first run)...")
simulator.warm_up()
print("JIT compilation complete.")

In [ ]:
# =============================================================================
# RUN SIMULATION
# =============================================================================

print("Running simulation...")
start_time = time.time()

for _ in range(1):
    # Noise is added inside JIT when include_noise=True
    response_signals, track_hits = simulator(deposit_data, key=jax.random.PRNGKey(42))

    # Wait for GPU completion
    if USE_BUCKETED:
        for key, val in response_signals.items():
            if val is not None:
                jax.block_until_ready(val[0])
    else:
        for arr in response_signals.values():
            if arr is not None:
                jax.block_until_ready(arr)

elapsed = time.time() - start_time
print(f"\nSimulation completed in {elapsed:.2f}s")
print(f"Throughput: {n_segments / elapsed:,.0f} segments/second")

## Process Response

Apply threshold and produce decomposed sparse output. Noise is already integrated in the JIT-compiled simulation when `include_noise=True`.

The output partitions surviving pixels into:
- **Signal entries** (first `n_signal`): pixels where the underlying signal is non-zero
- **Noise-only entries** (remaining): pixels that survived threshold due to noise alone

In [ ]:
# =============================================================================
# PROCESS RESPONSE: THRESHOLD + SPARSE DECOMPOSITION
# =============================================================================

print("Processing response signals...")
start_time = time.time()

# Noise already added inside JIT, so include_noise=False here
sparse_output = process_response(
    response_signals, detector_config,
    threshold_enc=THRESHOLD_ENC,
    include_noise=False,  # Noise already added in JIT
    key=None
)

elapsed_proc = time.time() - start_time
print(f"Processing completed in {elapsed_proc:.2f}s")

# Statistics
num_time_steps = detector_config['num_time_steps']
num_wires_actual = detector_config['num_wires_actual']
plane_names = ['U', 'V', 'Y']
side_names = ['East', 'West']

print(f"\nSparse output (threshold={THRESHOLD_ENC} e-, noise={'ON' if INCLUDE_NOISE else 'OFF'}):")
print(f"{'Plane':<15} {'Total':>10} {'Signal':>10} {'Noise-only':>12} {'Sparsity':>10}")
print("-" * 60)

for (si, pi), data in sparse_output.items():
    n_total = len(data['values'])
    n_sig = data['n_signal']
    n_noise = n_total - n_sig
    total_elements = int(num_wires_actual[si, pi]) * num_time_steps
    sparsity = (1 - n_total / total_elements) * 100 if total_elements > 0 else 0
    print(f"  {side_names[si]} {plane_names[pi]:<10} {n_total:>10,} {n_sig:>10,} {n_noise:>12,} {sparsity:>9.1f}%")

## Examine Outputs

In [ ]:
# =============================================================================
# EXAMINE TRACK_HITS (TRUTH OUTPUT)
# =============================================================================

total_hits = 0

if INCLUDE_TRACK_HITS:
    print("Truth hits (track_hits) per plane:")
    print(f"{'Plane':<15} {'Hits':>10}")
    print("-" * 28)

    for plane_key, results in track_hits.items():
        side_idx, plane_idx = plane_key
        num_hits = int(results['num_hits'])
        total_hits += num_hits
        print(f"{side_names[side_idx]} {plane_names[plane_idx]:<10} {num_hits:>10,}")

    print("-" * 28)
    print(f"{'Total':<15} {total_hits:>10,}")
    print(f"\nInput segments: {n_segments:,}")
    print(f"Output truth hits: {total_hits:,} (across all planes)")
else:
    print("Track labeling disabled (INCLUDE_TRACK_HITS=False)")

## Visualization

1. **Response signals**: Detector response (with or without noise)
2. **Truth hits**: Wire hits colored by deposited charge
3. **Track labels**: Wire hits colored by track ID

In [ ]:
# =============================================================================
# VISUALIZE RESPONSE SIGNALS
# =============================================================================

if INCLUDE_NOISE:
    vis_data = {k: (v['indices'], v['values']) for k, v in sparse_output.items()}
else:
    vis_data = extract_signal(sparse_output)

fig = visualize_wire_signals(vis_data, detector_config,
                             sparse_data=True, threshold_enc=THRESHOLD_ENC, gamma=0.2)

fig.savefig("plots/response_v4.png", dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# =============================================================================
# VISUALIZE TRUTH HITS (track_hits)
# =============================================================================

if INCLUDE_TRACK_HITS:
    def sparse_track_hits_to_dense(track_hits_dict, detector_config):
        """Convert sparse track hits to dense 2D arrays for visualization."""
        dense_signals = {}

        num_time_steps = detector_config['num_time_steps']
        num_wires_actual = detector_config['num_wires_actual']
        min_abs_indices = detector_config['min_wire_indices_abs']

        for plane_key, results in track_hits_dict.items():
            side_idx, plane_idx = plane_key
            num_wires = int(num_wires_actual[side_idx, plane_idx])
            min_wire_idx = int(min_abs_indices[side_idx, plane_idx])

            dense_array = jnp.zeros((num_wires, num_time_steps))

            if results['num_hits'] > 0:
                track_hits_data = results['hits_by_track'][:results['num_hits']]
                wire_indices_abs = track_hits_data[:, 0].astype(jnp.int32)
                time_indices = track_hits_data[:, 1].astype(jnp.int32)
                charges = track_hits_data[:, 2]

                wire_indices_rel = wire_indices_abs - min_wire_idx
                valid = (wire_indices_rel >= 0) & (wire_indices_rel < num_wires) & \
                        (time_indices >= 0) & (time_indices < num_time_steps)

                dense_array = dense_array.at[wire_indices_rel[valid], time_indices[valid]].add(charges[valid])

            dense_signals[plane_key] = dense_array

        return dense_signals

    print("Converting truth hits for visualization...")
    truth_hits_dense = sparse_track_hits_to_dense(track_hits, detector_config)

    fig_truth = visualize_diffused_charge(truth_hits_dense, detector_config, log_norm=True, threshold=50)
    fig_truth.suptitle(f'Event {EVENT_IDX} \u2014 Truth Hits ({total_hits:,} hits, before response convolution)',
                       fontsize=14, y=1.02)
    fig_truth.savefig("plots/truth_hits_v4.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("Skipping truth hits visualization (track labeling disabled)")

In [ ]:
# =============================================================================
# VISUALIZE TRACK LABELS
# =============================================================================

if INCLUDE_TRACK_HITS:
    top_tracks_by_charge = get_top_tracks_by_charge(track_hits, top_n=20)

    if top_tracks_by_charge:
        print("Top 10 tracks by total charge:")
        for i, (tid, charge) in enumerate(top_tracks_by_charge[:10]):
            print(f"  {i+1:2d}. Track {tid:4d}: {charge:12,.1f}")

    fig_tracks = visualize_track_labels(track_hits, detector_config, top_tracks_by_charge, max_tracks=15)
    fig_tracks.suptitle(f'Event {EVENT_IDX} \u2014 Track Labels ({n_unique_tracks:,} tracks)',
                        fontsize=14, y=1.02)
    fig_tracks.savefig("plots/track_labels_v4.png", dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
else:
    print("Skipping track label visualization (track labeling disabled)")

In [ ]:
# =============================================================================
# VISUALIZE SINGLE TRACK TRUTH HITS
# =============================================================================
# Pick a track by activity rank (1 = most active) OR by specific track ID.
# Set one to None to use the other.

TRACK_RANK = 2          # Nth most active track (1-indexed), or None
TRACK_ID = None         # Specific track ID, or None (ignored if TRACK_RANK is set)

if INCLUDE_TRACK_HITS:
    # Resolve which track ID to plot
    selected_tid = None
    if TRACK_RANK is not None:
        if TRACK_RANK <= len(top_tracks_by_charge):
            selected_tid, selected_charge = top_tracks_by_charge[TRACK_RANK - 1]
            print(f"Rank {TRACK_RANK} track: ID={selected_tid}, total charge={selected_charge:,.1f}")
        else:
            print(f"Rank {TRACK_RANK} exceeds available tracks ({len(top_tracks_by_charge)})")
    elif TRACK_ID is not None:
        selected_tid = TRACK_ID
        print(f"Selected track ID: {selected_tid}")

    if selected_tid is not None:
        # Filter labeled_hits to this track and build dense arrays
        num_time_steps_th = detector_config['num_time_steps']
        num_wires_actual_th = detector_config['num_wires_actual']
        min_abs_indices_th = detector_config['min_wire_indices_abs']

        single_track_dense = {}
        total_track_hits = 0

        for plane_key, results in track_hits.items():
            side_idx, plane_idx = plane_key
            num_wires = int(num_wires_actual_th[side_idx, plane_idx])
            min_wire_idx = int(min_abs_indices_th[side_idx, plane_idx])

            dense_array = jnp.zeros((num_wires, num_time_steps_th))
            num_labeled = int(results['num_labeled'])

            if num_labeled > 0:
                labeled = results['labeled_hits'][:num_labeled]
                # labeled columns: [track_id, wire_abs, time, charge]
                mask = labeled[:, 0].astype(jnp.int32) == selected_tid
                track_labeled = labeled[mask]

                if len(track_labeled) > 0:
                    wire_rel = track_labeled[:, 1].astype(jnp.int32) - min_wire_idx
                    time_idx = track_labeled[:, 2].astype(jnp.int32)
                    charges = track_labeled[:, 3]

                    valid = ((wire_rel >= 0) & (wire_rel < num_wires) &
                             (time_idx >= 0) & (time_idx < num_time_steps_th))
                    dense_array = dense_array.at[wire_rel[valid], time_idx[valid]].add(charges[valid])
                    total_track_hits += int(jnp.sum(mask))

            single_track_dense[plane_key] = dense_array

        print(f"Track {selected_tid}: {total_track_hits:,} labeled hits across all planes")

        fig_single = visualize_diffused_charge(single_track_dense, detector_config, log_norm=True, threshold=50)
        rank_str = f"Rank #{TRACK_RANK} — " if TRACK_RANK is not None else ""
        fig_single.suptitle(
            f'Event {EVENT_IDX} — {rank_str}Track {selected_tid} ({total_track_hits:,} hits)',
            fontsize=14, y=1.02
        )
        fig_single.savefig("plots/single_track_truth.png", dpi=300, bbox_inches='tight', facecolor='white')
        plt.show()
else:
    print("Skipping single track visualization (track labeling disabled)")

## Summary

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print("="*60)
print(" SIMULATION SUMMARY")
print("="*60)

print(f"\nInput:")
print(f"  Data file: {DATA_PATH}")
print(f"  Event: {EVENT_IDX}")
print(f"  Particle segments: {n_segments:,}")
print(f"  Unique tracks: {n_unique_tracks:,}")

print(f"\nProcessing:")
print(f"  East side: {n_east:,} segments")
print(f"  West side: {n_west:,} segments")
print(f"  Simulation time: {elapsed:.2f}s")
print(f"  Throughput: {n_segments / elapsed:,.0f} segments/second")

if INCLUDE_TRACK_HITS:
    print(f"\nOutput - Truth hits:")
    print(f"  Total hits: {total_hits:,} (with track_id labels)")

print(f"\nOutput - Processed response:")
total_pixels = sum(len(d['values']) for d in sparse_output.values())
total_signal = sum(d['n_signal'] for d in sparse_output.values())
total_noise = total_pixels - total_signal
total_dense = sum(int(num_wires_actual[s, p]) * num_time_steps
                  for s, p in sparse_output.keys())
print(f"  Surviving pixels: {total_pixels:,}")
print(f"  Signal pixels: {total_signal:,}")
print(f"  Noise-only pixels: {total_noise:,}")
print(f"  Sparsity: {(1 - total_pixels/total_dense)*100:.1f}%")

print(f"\nSettings:")
print(f"  Threshold: {THRESHOLD_ENC} e-")
print(f"  Noise: {'ON' if INCLUDE_NOISE else 'OFF'}")
print(f"  Track labeling: {'ON' if INCLUDE_TRACK_HITS else 'OFF'}")
if INCLUDE_NOISE:
    print(f"  Noise RMS (ADC): U/V={detector_config['noise_rms'][0,0]:.2f}, Y={detector_config['noise_rms'][0,2]:.2f}")